In [ ]:
%run ./1_config
from pyspark.sql import functions as F
from pyspark.sql.window import Window

class GoldBuilder:
    def __init__(self):
        self.c = conf
        self.v = spark.table(self.c.table_fqn(self.c.silver_visits_enriched))

    def suburb_interest(self):
        interstate = F.when(F.col("visitor_state") != F.col("listing_state"), F.lit(1)).otherwise(0)
        overseas = F.when(F.col("visitor_country") != "Australia", F.lit(1)).otherwise(0)
        df = (self.v.groupBy("suburb", "listing_state")
              .agg(F.count("*").alias("total_views"),
                   F.sum(interstate).alias("interstate_views"),
                   F.sum(overseas).alias("overseas_views"),
                   F.countDistinct("session_id").alias("unique_sessions")))
        df.write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.gold_suburb_interest))

    def property_type_by_suburb(self):
        w = Window.partitionBy("suburb").orderBy(F.desc("cnt"))
        df = (self.v.groupBy("suburb", "property_type").count().withColumnRenamed("count", "cnt")
              .withColumn("rn", F.row_number().over(w))
              .filter(F.col("rn") == 1)
              .select("suburb", F.col("property_type").alias("top_property_type"), "cnt"))
        df.write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.gold_property_type_by_suburb))

    def price_engagement(self):
        df = (self.v.groupBy("price_range")
              .agg(F.avg("view_duration_sec").alias("avg_view_time_sec"),
                   F.avg(F.when(F.col("enquiry_flag"), 1.0).otherwise(0.0)).alias("enquiry_rate"),
                   F.count("*").alias("views")))
        df.write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.gold_price_engagement))

    def conversion_gaps(self):
        df = (self.v.groupBy("suburb")
              .agg(F.count("*").alias("views"),
                   F.sum(F.when(F.col("enquiry_flag"), 1).otherwise(0)).alias("enquiries"))
              .withColumn("conversion_rate", F.col("enquiries") / F.col("views"))
              .orderBy(F.desc("views")))
        df.write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.gold_conversion_gaps))

    def property_trends(self):
        monthly = (self.v.withColumn("month", F.date_trunc("month", F.col("event_ts")))
                   .groupBy("suburb", "month").count().withColumnRenamed("count", "monthly_views"))
        w = Window.partitionBy("suburb").orderBy("month")
        df = monthly.withColumn("previous_month_views", F.lag("monthly_views").over(w))
        df.write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.gold_property_trends))

    def region_type_preference(self):
        df = (self.v.filter(F.col("visitor_state").isNotNull())
              .groupBy("visitor_state", "property_type")
              .count().withColumnRenamed("count", "views"))
        w = Window.partitionBy("visitor_state").orderBy(F.desc("views"))
        df = df.withColumn("rn", F.row_number().over(w)).filter(F.col("rn") == 1).select(
            F.col("visitor_state").alias("visitor_region"),
            F.col("property_type").alias("preferred_type"), "views")
        df.write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.gold_region_type_preference))

    def repeat_interest(self):
        df = (self.v.groupBy("session_id", "suburb")
              .agg(F.count("*").alias("repeated_views"),
                   F.max("enquiry_flag").alias("ever_enquired"))
              .filter(F.col("repeated_views") >= 5)
              .orderBy(F.desc("repeated_views")))
        df.write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.gold_repeat_interest))

    def interstate_flow(self):
        df = (self.v.filter(F.col("visitor_state") != F.col("listing_state"))
              .groupBy(F.col("visitor_state").alias("visitor_state"), F.col("listing_state").alias("viewed_state"))
              .agg(F.count("*").alias("views"), F.countDistinct("session_id").alias("sessions")))
        df.write.format("delta").mode("overwrite").saveAsTable(self.c.table_fqn(self.c.gold_interstate_flow))

    def run(self):
        self.suburb_interest()
        self.property_type_by_suburb()
        self.price_engagement()
        self.conversion_gaps()
        self.property_trends()
        self.region_type_preference()
        self.repeat_interest()
        self.interstate_flow()
        print("✅ Gold tables built")

    def validate(self):
        for name in [self.c.gold_suburb_interest, self.c.gold_conversion_gaps, self.c.gold_property_trends]:
            print(f"🔎 {name}: {spark.table(self.c.table_fqn(name)).count()} rows")

GoldBuilder().run()
GoldBuilder().validate()

